# Quick Start: ART in 5 Minutes

**Time:** < 2 minutes to read, 5 minutes to run

Set up ART, connect to an MCP server, and run your first training step.

---

## Prerequisites

```bash
pip install art-trainer fastmcp
export OPENAI_API_KEY=sk-...  # for RULER judge
```

In [ ]:
# Step 1: Create a sample database
import sqlite3

conn = sqlite3.connect("quickstart.db")
conn.executescript("""
    CREATE TABLE IF NOT EXISTS products (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        price REAL NOT NULL,
        stock INTEGER NOT NULL
    );
    
    DELETE FROM products;
    INSERT INTO products VALUES
        (1, 'Laptop Pro', 'Electronics', 1299.99, 45),
        (2, 'Wireless Mouse', 'Electronics', 29.99, 200),
        (3, 'Standing Desk', 'Furniture', 599.99, 30),
        (4, 'Monitor 4K', 'Electronics', 449.99, 75),
        (5, 'Ergonomic Chair', 'Furniture', 399.99, 50);
""")
conn.commit()
conn.close()
print("Database created with 5 products.")

In [ ]:
# Step 2: Build MCP server
from fastmcp import FastMCP
import sqlite3

mcp = FastMCP("product-db")

@mcp.tool()
def list_tables() -> str:
    """List all tables in the database."""
    conn = sqlite3.connect("quickstart.db")
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    conn.close()
    return "\n".join(t[0] for t in tables)

@mcp.tool()
def run_query(sql: str) -> str:
    """Run a SELECT query on the database."""
    if not sql.strip().upper().startswith("SELECT"):
        return "Error: Only SELECT queries allowed."
    conn = sqlite3.connect("quickstart.db")
    try:
        rows = conn.execute(sql).fetchall()
        return "\n".join(str(r) for r in rows)
    except sqlite3.Error as e:
        return f"Error: {e}"
    finally:
        conn.close()

print("MCP server defined with 2 tools.")
print(f"Tools: {[t.name for t in mcp.list_tools()]}")

In [ ]:
# Step 3: Simulate what ART does
# (In production, ART handles this with a real model + vLLM backend)

import numpy as np

# Simulated scenarios
scenarios = [
    "What is the most expensive product?",
    "How many electronics products are in stock?",
    "What is the total value of all inventory?",
]

# Simulated rollout results (4 attempts per scenario)
# In reality, the model generates these via tool calls
scenario_rewards = [
    [0.2, 0.8, 0.5, 0.9],  # scenario 1: attempts of varying quality
    [0.1, 0.6, 0.7, 0.4],  # scenario 2
    [0.3, 0.3, 0.8, 0.6],  # scenario 3
]

print("Simulated GRPO Training Step")
print("=" * 50)

for scenario, rewards in zip(scenarios, scenario_rewards):
    r = np.array(rewards)
    advantages = (r - r.mean()) / (r.std() + 1e-8)
    
    print(f"\nScenario: {scenario}")
    print(f"  Rewards:    {rewards}")
    print(f"  Advantages: {[f'{a:+.2f}' for a in advantages]}")
    print(f"  Best attempt: #{np.argmax(rewards)+1} (reward={max(rewards)})")

## What's Next

With a real ART setup:

```python
from art import ART

# Initialize model
model = ART(
    model="Qwen/Qwen2.5-3B",
    reward_fn="ruler",
    judge_model="openai/o4-mini",
)

# Connect MCP tools
tools = model.connect_mcp("http://localhost:8000")

# Generate scenarios
scenarios = model.generate_scenarios(tools, n=20)

# Train
model.train(scenarios=scenarios, steps=50, group_size=4)
```

See **Module 02** for full ART framework setup and **Module 06** for the complete text-to-SQL project.